# PRE-05 · Practical Python, Pandas, Debugging, and Tests
### Prerequisite Phase — Bridge from small code cells to data work

**Prerequisite:** PRE-03. **Estimated time:** 5–7 hours including practice.
This module exists because knowing NumPy syntax is not yet enough to debug a
data workflow. Complete it before FND-03; experienced Python users may pass the
readiness gate instead.


> **Canonical learner route · module PRE-05 of 65**
>
> Required prerequisites: **PRE-03** · Previous: **PRE-04** ·
> Next after mastery: **FND-01** · Expected first-pass workload:
> **2–4 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives

Write and test small functions; use lists, dictionaries, conditions, loops, and
exceptions; read tracebacks; load and inspect tabular data; select, filter,
group, join, and validate pandas data; plot a distribution; and distinguish a
code failure from a data-contract failure.


## Student Lesson Companion · PRE-05 · Practical Python, Pandas, Debugging, and Tests

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |

### Code walkthrough and expected-result contract

For the first implementation, annotate: inputs → shapes/units → initialization →
central computation → intermediate output → final output → verification. Before
execution, record the expected value, range, or shape and what it would mean.

Distinguish these outcomes:

| Outcome | Interpretation | Next action |
|---|---|---|
| Exception, non-finite value, impossible shape | Code or data-contract failure | Inspect the first violated boundary |
| Output has valid type/shape but weak metric | Experiment ran; method may be poor | Diagnose data, assumptions, and baseline comparison |
| Strong metric on training data only | Insufficient evidence | Evaluate with the declared validation design |
| Expected output on untouched data | Supports one scoped claim | Record limitations; do not generalize beyond evidence |

### Debugging table

| Symptom | Likely cause | Inspect | Scoped fix |
|---|---|---|---|
| Shape/type error | Interface mismatch | Shapes, dtypes, feature names | Repair the boundary, not downstream symptoms |
| NaN/Inf or divergence | Invalid input or unstable update | Raw values, loss, gradients, learning rate | Clean/validate input or change one optimization control |
| Implausibly strong result | Leakage or invalid split | Fit boundaries, timestamps, duplicate entities | Rebuild the evaluation path before tuning |
| Different repeated result | Uncontrolled state | Seeds, data order, train/eval mode, versions | Record and control randomness intentionally |
| Plausible output but poor result | Wrong assumptions or representation | Baseline, slices, residuals/errors | Prefer a justified alternative; do not debug valid code as broken |


## 2 · Historical Motivation

Real ML work is mostly ordinary programs around numerical kernels: loading data,
validating assumptions, transforming columns, logging decisions, and testing
interfaces. A model cannot rescue a pipeline that silently selects the wrong row,
changes units, or catches every exception.


## 3 · Intuition and Visual Understanding

Treat code as small contracts:

```text
input assumptions → function → output guarantees
       ↓                          ↓
   assertions                  tests
```

Read a traceback from the final line upward: exception type and message first,
then the last frame you own, then inspect the values and shapes at that boundary.


## 4 · Mathematical Foundations

A boolean mask is an indicator vector. If $m_i\in\{0,1\}$ marks whether row $i$
passes a condition, then $\sum_i m_i$ is the selected-row count. For mask
`[True, False, True]`, the count is 2. This connection explains filtering,
missing-value counts, and group aggregation without introducing new mathematics.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

records = [
    {"sample_id": "a", "group": "control", "value": 2.0},
    {"sample_id": "b", "group": "treatment", "value": 5.0},
    {"sample_id": "c", "group": "treatment", "value": None},
]
frame = pd.DataFrame(records)
print(frame)


## 5 · Manual Implementation from Scratch

Begin with plain Python before pandas. A function should validate what it cannot
safely infer and raise an informative error rather than return plausible nonsense.


In [ ]:
def safe_mean(values):
    cleaned = [float(value) for value in values if value is not None]
    if not cleaned:
        raise ValueError("safe_mean needs at least one non-missing value")
    return sum(cleaned) / len(cleaned)

assert safe_mean([2, None, 4]) == 3.0
try:
    safe_mean([None])
except ValueError as error:
    print(type(error).__name__, str(error))


## 6 · Visualization

A plot is a diagnostic, not decoration. State the question first, label axes and
units, then explain one visible pattern and one thing the plot cannot establish.


In [ ]:
frame["value"].plot(kind="hist", bins=4, title="Observed value distribution")
plt.xlabel("value (arbitrary units)")
plt.tight_layout()
plt.show()


## 7 · Failure Modes and Common Mistakes

- Catching `Exception` and discarding the error.
- Mutating a DataFrame without knowing whether a view or copy is involved.
- Comparing missing values with `== None` instead of using `isna`.
- Selecting columns by position when names are the contract.
- Joining on a non-unique key and silently multiplying rows.
- Treating `object` dtype as a meaningful schema.


## 8 · Library Implementation

Use pandas operations deliberately: inspect schema, select named columns, filter
with masks, aggregate with `groupby`, merge with cardinality validation, and make
assumptions executable with assertions.


In [ ]:
required = {"sample_id", "group", "value"}
assert required.issubset(frame.columns)
assert frame["sample_id"].is_unique
summary = frame.groupby("group", dropna=False)["value"].agg(["count", "mean"])
print(summary)

metadata = pd.DataFrame({"sample_id": ["a", "b", "c"], "site": ["x", "x", "y"]})
joined = frame.merge(metadata, on="sample_id", validate="one_to_one")
assert len(joined) == len(frame)


## 9 · Realistic Case Study

Given a CSV of measurements, write a loader that checks required columns, unique
IDs, numeric conversion, missingness, and allowed labels. Return a clean table and
a report; never silently delete invalid rows. Lesson FND-03 will turn this table
into a leakage-safe modeling workflow.


## 10 · Production and Learning Considerations

Keep environments reproducible, use relative paths from the repository root,
avoid secrets in notebooks, and move reusable logic into importable functions.
Tests should use tiny deterministic data and cover valid input plus one failure.


## 11 · Tradeoff Analysis

Vectorized pandas is concise and fast for ordinary tables; explicit loops can be
clearer for stateful logic. Assertions are excellent learning checks but public
input validation should raise intentional exceptions. Notebooks aid exploration;
modules and tests aid reuse.


## 12 · Readiness and Interview Preparation

You are ready when you can debug a `KeyError`, explain DataFrame shape and dtypes,
prevent a many-to-many join, write a pure function with a test, and load a table
without relying on hidden notebook state.


## 13 · Teach-Back

Explain the difference between a list, dictionary, NumPy array, and DataFrame.
Then describe how you would investigate “the row count doubled after enrichment”
and why restarting the kernel is a useful reproducibility test.


## 14 · Exercises, Self-Check, and Solutions

**Worked example:** the merge above uses `validate="one_to_one"`; duplicate a
metadata key and observe the intentional error.

**Guided practice (30 min):** add a `unit` column, reject values whose unit is
neither `mg` nor `g`, and convert grams to milligrams. Hint: write one conversion
function and use `.apply`. Self-check: `2 g → 2000 mg`.

**Independent practice (45 min):** create a five-row CSV in a temporary directory,
load it, check required columns and unique IDs, and return a missingness report.
Self-check: deliberately remove one column and confirm an informative failure.

**Challenge (60 min):** join measurements to entity metadata while guaranteeing
row count and key cardinality; plot one labelled distribution by group.

<details><summary><strong>Solution and scoring rubric</strong></summary>

A complete solution uses named columns, `isna`, `is_unique`, `merge(validate=...)`,
an explicit exception, and at least two assertions. Award 2 points for conversion,
3 for the loader/report, 3 for safe joining, and 2 for explanation and tests.
Common mistakes: hidden global variables, broad exception handling, positional
columns, and accepting a multiplied join. **Readiness threshold: 8/10.**
</details>


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **PRE-05 · Practical Python, Pandas, Debugging, and Tests** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember PRE-05 · Practical Python, Pandas, Debugging, and Tests: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · PRE-05

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
